# Embedding clusters

The same 100k-complaint sample as `03_tfidf_topics`, clustered on sentence
embeddings (`nhtsa.embed.embed_sample`, cached after the first run) instead of
TF-IDF:

- **k-means on embeddings** with 25 clusters: the like-for-like comparison with
  the TF-IDF baselines.
- **UMAP + HDBSCAN**: reduce to 5 dimensions, then density clustering. It
  chooses its own number of clusters and leaves complaints that fit nowhere
  unassigned.

Clusters are described by their most distinctive words (class-based TF-IDF,
`nhtsa.topics.cluster_term_weights`) and scored with `nhtsa.evaluate.scores`.

In [1]:
import warnings

import numpy as np
import polars as pl
from IPython.display import display
from sklearn.cluster import HDBSCAN, KMeans
from umap import UMAP

from nhtsa.dataset import sample_complaints
from nhtsa.embed import DEFAULT_MODEL, embed_sample
from nhtsa.evaluate import scores
from nhtsa.notebook import (
    cluster_names,
    cluster_table,
    label_share_chart,
    show_texts,
    year_share_chart,
)
from nhtsa.topics import (
    cluster_term_weights,
    lsa_kmeans,
    nmf_topics,
    strongest_topic,
    tfidf_features,
    top_terms,
)

pl.Config.set_tbl_rows(80).set_fmt_str_lengths(120).set_float_precision(3);

In [2]:
sample = sample_complaints()
texts = sample["TEXT"].to_list()
embeddings = embed_sample()
print(f"{DEFAULT_MODEL}: {embeddings.shape[0]:,} x {embeddings.shape[1]} embeddings.")


def describe(clusters):
    """Each cluster's ten most distinctive words."""
    return top_terms(*cluster_term_weights(texts, clusters))


def show_central(clusters, cluster, n=6):
    """The n complaints closest to the cluster's mean embedding."""
    members = np.flatnonzero(clusters == cluster)
    center = embeddings[members].mean(axis=0)
    for i in members[(embeddings[members] @ center).argsort()[::-1][:n]]:
        r = sample.row(int(i), named=True)
        display(
            show_texts(
                f"{r['ODINO']} · {r['YEAR']} · {', '.join(r['LABELS'])}", r["TEXT"]
            )
        )

BAAI/bge-base-en-v1.5: 100,000 x 768 embeddings.


## k-means on embeddings

`top_label` / `label_share`: the most common component label among the
cluster's single-label complaints, and its share.

In [3]:
N_CLUSTERS = 25  # the number of component labels

kmeans_clusters = KMeans(N_CLUSTERS, random_state=0, n_init=3).fit_predict(embeddings)
kmeans_words = describe(kmeans_clusters)
kmeans_names = cluster_names(kmeans_words)
cluster_table(kmeans_clusters, sample["LABELS"], kmeans_words)

name,complaints,top_label,label_share,top_words
str,u32,str,f64,str
"""00 vehicle, driving, mph""",2940,"""powertrain""",0.204,"""vehicle, driving, mph, consumer, causing, control, stop, steering, brakes, speed"""
"""01 recall, parts, safety""",5093,"""air_bags""",0.321,"""recall, parts, safety, vehicle, consumer, notice, campaign, takata, received, available"""
"""02 frame, rear, broke""",3957,"""suspension""",0.355,"""frame, rear, broke, wheel, rusted, axle, suspension, vehicle, rust, ball"""
"""03 wipers, windshield, wiper""",914,"""visibility""",0.971,"""wipers, windshield, wiper, windshield wipers, windshield wiper, rain, wiper motor, motor, working, work"""
"""04 engine, oil, miles""",4330,"""engine""",0.788,"""engine, oil, miles, car, vehicle, problem, dealership, warranty, light, issue"""
"""05 car, stop, vehicle""",5816,"""speed_control""",0.233,"""car, stop, vehicle, speed, brake, driving, mph, control, pedal, traffic"""
"""06 car, start, battery""",6875,"""electrical""",0.468,"""car, start, battery, problem, engine, light, vehicle, issue, driving, turn"""
"""07 windshield, glass, window""",3670,"""visibility""",0.568,"""windshield, glass, window, crack, sunroof, cracked, dashboard, dash, driving, shattered"""
"""08 smoke, vehicle, caught""",2548,"""electrical""",0.395,"""smoke, vehicle, caught, flames, hood, department, engine, parked, car, coming"""


In [4]:
label_share_chart(kmeans_clusters, sample["LABELS"], kmeans_names)

alt.Chart(...)

In [5]:
year_share_chart(kmeans_clusters, sample["YEAR"], kmeans_names)

alt.Chart(...)

## UMAP + HDBSCAN

HDBSCAN works poorly in 768 dimensions, so UMAP first reduces the embeddings
to 5. `min_cluster_size` sets the smallest group that counts as a cluster;
smaller values give more, finer clusters (about 145 at 100, about 43 at 500).

In [6]:
# UMAP warns that its default layout pipeline will change in a future release.
warnings.filterwarnings("ignore", category=FutureWarning, module="umap")

reduced = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=0,
    n_jobs=1,  # a fixed random_state makes UMAP single-threaded anyway
).fit_transform(embeddings)

In [7]:
hdbscan_clusters = HDBSCAN(min_cluster_size=500, min_samples=10, copy=True).fit_predict(
    reduced
)
hdbscan_words = describe(hdbscan_clusters)
hdbscan_names = cluster_names(hdbscan_words)
print(
    f"{len(hdbscan_words)} clusters; "
    f"{(hdbscan_clusters < 0).mean():.1%} of complaints unassigned."
)
cluster_table(hdbscan_clusters, sample["LABELS"], hdbscan_words)

43 clusters; 28.9% of complaints unassigned.


name,complaints,top_label,label_share,top_words
str,u32,str,f64,str
"""00 campaign, campaign air, recall received""",973,"""air_bags""",0.988,"""campaign, campaign air, recall received, takata recall, recall, takata, repair, air bags, bags, recall repair"""
"""01 wipers, windshield, wiper""",929,"""visibility""",0.977,"""wipers, windshield, wiper, windshield wipers, windshield wiper, rain, wiper motor, motor, working, visibility"""
"""02 airbag, bag, air bag""",4525,"""air_bags""",0.910,"""airbag, bag, air bag, air, light, recall, passenger, takata, seat, takata recall"""
"""03 campaign, campaign air, repair""",831,"""air_bags""",0.964,"""campaign, campaign air, repair, air bags, bags, recall repair, notification nhtsa, notification, reasonable time, reason…"
"""04 deploy, air, deployed""",3418,"""air_bags""",0.756,"""deploy, air, deployed, vehicle, bags, air bags, did deploy, injuries, driver, did"""
"""05 water, mold, leak""",559,"""structure""",0.598,"""water, mold, leak, leaking, rains, rain, windshield, carpet, leaks, sunroof"""
"""06 mirror, dashboard, glare""",656,"""visibility""",0.584,"""mirror, dashboard, glare, melting, mirrors, windshield, sticky, view, dash, visibility"""
"""07 dash, dashboard, cracked""",608,"""structure""",0.777,"""dash, dashboard, cracked, airbag, cracks, crack, cracking, passenger, dodge, passenger airbag"""
"""08 windshield, crack, cracked""",572,"""visibility""",0.966,"""windshield, crack, cracked, subaru, windshield cracked, rock, chip, glass, small, crack windshield"""


In [8]:
label_share_chart(hdbscan_clusters, sample["LABELS"], hdbscan_names)

alt.Chart(...)

In [9]:
year_share_chart(hdbscan_clusters, sample["YEAR"], hdbscan_names)

alt.Chart(...)

## Scores

The TF-IDF baselines from `03_tfidf_topics` are recomputed here so every
method is scored on the same rows. The random row is a reference: what the
scores look like with no structure.

HDBSCAN's NMI and ARI are not directly comparable with the other rows: it
picks its own number of clusters, and its scores cover only the complaints it
assigned (see `unassigned`).

In [10]:
tfidf, _ = tfidf_features(texts)
methods = {
    "NMF topics": strongest_topic(nmf_topics(tfidf, N_CLUSTERS)[0]),
    "LSA + k-means": lsa_kmeans(tfidf, N_CLUSTERS),
    "embeddings + k-means": kmeans_clusters,
    "UMAP + HDBSCAN": hdbscan_clusters,
    "random (reference)": np.random.default_rng(0).integers(
        0, N_CLUSTERS, sample.height
    ),
}
pl.DataFrame(
    [
        {
            "method": name,
            "clusters": len(np.unique(c[c >= 0])),
            **scores(c, sample["LABELS"], sample["YEAR"].to_numpy()),
        }
        for name, c in methods.items()
    ]
)

method,clusters,nmi,ari,coverage,era_accuracy,unassigned
str,i64,f64,f64,f64,f64,f64
"""NMF topics""",25,0.350,0.228,0.740,0.594,0.000
"""LSA + k-means""",25,0.366,0.189,0.740,0.570,0.000
"""embeddings + k-means""",25,0.442,0.271,0.740,0.566,0.000
"""UMAP + HDBSCAN""",43,0.581,0.317,0.542,0.631,0.289
"""random (reference)""",25,0.001,0.000,0.740,0.511,0.000


## Read a cluster

The complaints closest to the centre of a cluster. Change `CLUSTERS` and
`CLUSTER` (the number in the cluster's name) and re-run the cell.

In [11]:
CLUSTERS, NAMES = kmeans_clusters, kmeans_names  # or hdbscan_clusters, hdbscan_names
CLUSTER = 0

print(NAMES[CLUSTER])
show_central(CLUSTERS, CLUSTER)

00 vehicle, driving, mph
